# Week 2, day 5 (morning) — Worksheet 06 SOLUTIONS: combined challenges   (L05)

Every cell below was executed on the same Python the lab ships (3.13), and the
quoted output is what it actually printed.

There is more than one right way to write most of these. If your numbers match
and your code is readable, you are done — the solutions here favour the plainest
loop over the cleverest one-liner.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 06 — Combined challenges. Run this once.
#
# One morning of clickstream, already sorted by time.
# (minutes since 09:00, user, action)
events = [
    (0,  "u1", "view"),
    (3,  "u1", "view"),
    (5,  "u2", "view"),
    (9,  "u1", "cart"),
    (12, "u1", "purchase"),
    (48, "u1", "view"),
    (50, "u3", "view"),
    (52, "u2", "cart"),
    (55, "u1", "view"),
    (61, "u2", "purchase"),
    (95, "u3", "view"),
    (97, "u3", "cart"),
]

SESSION_GAP = 30      # a gap longer than this starts a new session

print(len(events), "events, from minute", events[0][0], "to", events[-1][0])

PART A — Counting things

### Question 1

The shape of the data. -> `12 events`, `3 users: ['u1', 'u2', 'u3']`, `3 actions: ['cart', 'purchase', 'view']`.

`{user for minute, user, action in events}` unpacks each tuple into three
names and keeps only the middle one. Naming the parts you do not use is
clearer than `e[1]` everywhere, and it fails loudly if a row ever has four
fields instead of three.

The sets are printed through `sorted()`, which returns a **list** in a
fixed order. Print the set itself and the arrangement will differ between
runs — if a number in a report comes from a set, sort it before showing it.

In [ ]:
users = {user for minute, user, action in events}
actions = {action for minute, user, action in events}

print(len(events), "events")
print(len(users), "users:", sorted(users))
print(len(actions), "actions:", sorted(actions))

### Question 2

Counting two ways in one pass. -> `{'u1': 6, 'u2': 3, 'u3': 3}` and `{'view': 7, 'cart': 3, 'purchase': 2}`.

Both dictionaries are filled by the same loop. One pass, two answers —
worth doing deliberately, because the instinct is to write two loops.

The totals check out: 6 + 3 + 3 = 12, and 7 + 3 + 2 = 12. Any
counting loop over the whole dataset should add back up to the row count,
and if it does not you have either skipped rows or double-counted them.
It takes one line to check and it catches a whole class of bug.

In [ ]:
per_user = {}
per_action = {}

for minute, user, action in events:
    if user in per_user:
        per_user[user] = per_user[user] + 1
    else:
        per_user[user] = 1
    if action in per_action:
        per_action[action] = per_action[action] + 1
    else:
        per_action[action] = 1

print(per_user)
print(per_action)

### Question 3

First and last sighting. -> `first_seen` is `{'u1': 0, 'u2': 5, 'u3': 50}`, `last_seen` is `{'u1': 55, 'u2': 61, 'u3': 97}`. Spans: u1 `55`, u2 `56`, u3 `47`.

The two lines look almost identical and behave in opposite ways. `first_seen`
is guarded by `if user not in first_seen:` so it is written **once** and
never touched again. `last_seen` is unguarded, so every event overwrites
it and it ends up holding the final one.

Guard / do-not-guard is the whole difference between *first* and *last*, and
it only works because the events are sorted by time. On unsorted data both
lines are wrong and neither complains — which is why the setup cell says
"already sorted by time" and why you should check that claim on real data
rather than trusting it.

In [ ]:
first_seen = {}
last_seen = {}

for minute, user, action in events:
    if user not in first_seen:
        first_seen[user] = minute      # only the first time we meet them
    last_seen[user] = minute           # overwritten every time -- ends on the last

print(first_seen)
print(last_seen)

for user in sorted(first_seen):
    print(user, "span", last_seen[user] - first_seen[user], "minutes")

PART B — Looking at neighbours

### Question 4

Gaps between neighbours. -> `[3, 2, 4, 3, 36, 2, 2, 3, 6, 34, 2]`, `11 gaps from 12 events`. Biggest is `36`, between minute `12` and `48`.

Comparing an item with its neighbour is the one job `for row in events:`
cannot do — the loop hands you one row at a time and no way to reach the
previous one. `range(1, len(events))` gives you the position, so you can
look at `[i]` and `[i - 1]` together.

Starting at 1 is not an off-by-one; it is the point. The first event has
no predecessor, so *n* events give *n − 1* gaps. If your gaps list is the
same length as your data, you have invented a gap somewhere.

`gaps.index(biggest)` finds the **first** position holding that value. With
two equally large gaps it would silently report only the earlier one.

In [ ]:
gaps = []
for i in range(1, len(events)):
    gaps.append(events[i][0] - events[i - 1][0])

print(gaps, len(gaps), "gaps from", len(events), "events")

biggest = max(gaps)
where = gaps.index(biggest)
print("biggest gap", biggest, "between minute",
      events[where][0], "and", events[where + 1][0])

### Question 5

Sessionising. -> `{'u1': 2, 'u2': 2, 'u3': 2}`, and `6 sessions from 12 events`.

Two dictionaries again, and the second one is the interesting half.
`last_minute` is not part of the answer — it is **state**: the thing the
loop has to remember about each user in order to judge the next event.
Every per-group calculation over a sequence looks like this.

The three branches map to the three cases exactly: never seen this user
(start at session 1), seen them but the gap is too long (new session), seen
them recently (nothing to do). The third case has no branch at all, which
is correct — the only thing that always happens is the `last_minute` update
at the bottom.

One gap per user was over 30 minutes, so every user has exactly two
sessions. That is a coincidence of this data, not a rule.

In [ ]:
sessions = {}
last_minute = {}

for minute, user, action in events:
    if user not in last_minute:
        sessions[user] = 1                       # first event: session one
    elif minute - last_minute[user] > SESSION_GAP:
        sessions[user] = sessions[user] + 1      # too long a gap: new session
    last_minute[user] = minute

print(sessions)
print(sum(sessions.values()), "sessions from", len(events), "events")

PART C — Searching

### Question 6

Two searches. -> `first purchase at minute 12 by u1`, then `u3 never purchased`.

The first loop stops at minute 12 and never looks at the other seven
events. The second walks the whole log, finds nothing, falls out of the
bottom, and the `else` fires.

A search has **two** outcomes and both need code. Without the `else` the
second loop would print nothing at all — and "no output" is
indistinguishable from "the cell did not run", which is how a missing
result turns into a wrong conclusion.

Note how the compound condition reads: `user == "u3" and action ==
"purchase"`. Two facts about the same row, tested together, exactly as in
worksheet 01 Q7.

In [ ]:
for minute, user, action in events:
    if action == "purchase":
        print("first purchase at minute", minute, "by", user)
        break

for minute, user, action in events:
    if user == "u3" and action == "purchase":
        print("u3 first purchased at minute", minute)
        break
else:
    print("u3 never purchased")

### Question 7

Filter, then map. -> `['view', 'view', 'cart', 'purchase', 'view', 'view'] 6`, then twelve labels, six `first half` and six `second half`, length `12`.

**Six against twelve** — that is worksheet 03 Q5 in real use. The filter
form (`if` at the end) dropped the events belonging to other users; the map
form (`if` at the front) kept every event and changed what each one became.

The six u1 actions agree with Q2's count of 6, which is the kind of
cross-check worth making automatically.

And `halves` is in the same order as `events`, so it lines up position by
position with the original list — that is the property that makes the map
form useful. Feed it to `zip(events, halves)` and nothing has to be
re-sorted.

In [ ]:
u1_actions = [action for minute, user, action in events if user == "u1"]
print(u1_actions, len(u1_actions))

halves = ["first half" if minute < 50 else "second half"
          for minute, user, action in events]
print(halves, len(halves))

### Question 8

The funnel. -> u1 and u2 both `['cart', 'purchase', 'view'] -- completed: True`; u3 `['cart', 'view'] -- completed: False`. `2 of 3 users completed the funnel`.

A dictionary whose values are **sets** is the standard shape for "which
things did each group do". The set collapses u1's six events into three
distinct actions for free — that is the same deduplication as `set(...)` on
a list, happening incrementally.

`did[user] = set()` before the first `.add()` is the same guard as Q2's
counting `else`: you cannot add to a set that does not exist yet.

What this does **not** tell you is order. u3 has `cart` and `view`, and
this code cannot say whether they viewed before carting. A funnel usually
means the steps happened in sequence, and a set has thrown that away —
true of every question on this sheet that uses one.

In [ ]:
did = {}
for minute, user, action in events:
    if user not in did:
        did[user] = set()
    did[user].add(action)

completed = 0
for user in sorted(did):
    full = "view" in did[user] and "cart" in did[user] and "purchase" in did[user]
    print(user, sorted(did[user]), "-- completed:", full)
    if full:
        completed = completed + 1

print(completed, "of", len(did), "users completed the funnel")

PART D — Carrying state between passes

### Question 9

The longest run. -> `longest run: 3 events by u1 starting at minute 9`.

u1 at minutes 9, 12 and 48 — three consecutive log lines with no other
user between them. (They span a 36-minute gap, so it is a run in the log,
not a run in time. Which one you wanted is worth deciding before you write
the loop.)

Six variables, and they split into two groups: `current_*` describes the
run being walked right now, `best_*` remembers the winner so far. That
separation is the pattern — the current run resets constantly, the best one
only ever improves.

The `if current_run > best_run:` sits **outside** the if/else on purpose, so
it is checked on every pass. Put it inside the first branch and a
single-event run could never win; put it after the loop and you would only
ever see the last run.

Strictly `>` and not `>=`: ties keep the earlier run. That is a decision,
not a detail — with `>=` the answer would be a different user on data with
two runs of three.

In [ ]:
current_user = None
current_run = 0
best_run = 0
best_user = None
best_start = None
run_start = None

for i in range(len(events)):
    minute, user, action = events[i]
    if user == current_user:
        current_run = current_run + 1
    else:
        current_user = user
        current_run = 1
        run_start = minute
    if current_run > best_run:
        best_run = current_run
        best_user = current_user
        best_start = run_start

print("longest run:", best_run, "events by", best_user,
      "starting at minute", best_start)

PART E — The number that does not mean what it says

### Question 10

Three conversion rates. -> purchases `2`; per event `0.16666666666666666`; per user `0.6666666666666666`; per session `0.3333333333333333`.

One numerator, three denominators, and the top figure is **four times** the
bottom one. Every division is correct. Every one of them could be pasted
into a slide under the words "conversion rate".

- **Per event** (17%) is the one to reject outright. Its denominator grows
  every time somebody clicks, so a site that made browsing easier would
  report a *falling* conversion rate on identical sales.
- **Per user** (67%) counts a person once no matter how many times they
  came back without buying. It ignores u1's second visit and u3's second
  visit entirely — the two visits that converted nothing.
- **Per session** (33%) is what the phrase normally means: of six visits,
  two ended in a purchase.

So report the third — and say "per session" out loud when you do, because
the number is meaningless without its denominator.

Notice what did **not** happen: no error, no warning, no hint from Python
that two of these are answers to questions nobody asked. Getting the loop
to run was the easy half. Knowing which denominator you are entitled to is
the half that matters, and it is the same lesson the SQL sessions ended on.

In [ ]:
purchases = 0
for minute, user, action in events:
    if action == "purchase":
        purchases = purchases + 1

total_sessions = sum(sessions.values())

print("purchases:", purchases)
print("per event:  ", purchases / len(events))
print("per user:   ", purchases / len(did))
print("per session:", purchases / total_sessions)

# All three are real divisions of real counts. "Conversion rate" almost
# always means per SESSION -- a visit that ended in a purchase -- so that is
# the one to report, and only after saying so out loud.
#
# Per event is meaningless: it just falls as people click more. Per user
# flatters the number by ignoring the visits that converted nothing. Neither
# is wrong arithmetic; both answer a question nobody asked.